In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
    # [results_1l, results_2l, results_3l],
    # ignore_index=True
# )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_SuperZZ1_theta,MSE_SuperZZ1_theta,R2_SuperZZ2_theta,MSE_SuperZZ2_theta,...,R2_ZZx2_theta,MSE_ZZx2_theta,R2_ZZxReto_theta,MSE_ZZxReto_theta,R2_ZZy1_theta,MSE_ZZy1_theta,R2_ZZy2_theta,MSE_ZZy2_theta,R2_semiCirc_theta,MSE_semiCirc_theta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed8013,[1],0.3,0.7,0.01,8013,0.747054,0.542616,-1.504874,0.497839,...,-5.222703,0.095727,-0.886495,0.401239,-23.238633,0.005958,0.716991,0.548866,-9.085380,0.216097
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed3064,[1],0.3,0.7,0.01,3064,0.812422,0.584540,-1.834002,0.523034,...,-4.878421,0.174038,-0.568868,0.493377,-29.037565,0.002655,0.729441,0.577519,-17.335011,0.034071
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed3365,[1],0.3,0.7,0.01,3365,0.896052,0.588125,-3.229715,0.467545,...,-4.430127,0.261065,0.122323,0.527371,-19.539860,0.139193,0.118744,0.541978,-14.842405,0.134502
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed7022,[1],0.3,0.7,0.01,7022,0.784890,0.546470,-1.872818,0.489716,...,-4.790728,0.137517,-0.683701,0.423289,-20.972692,0.045878,0.578899,0.546712,-8.828469,0.232660
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed4237,[1],0.3,0.7,0.01,4237,0.711963,0.532453,-1.212054,0.500102,...,-5.451293,0.063867,-1.032297,0.377910,-23.792744,-0.016880,0.776727,0.545926,-8.526380,0.220104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2008,model_arch100_r0.9_Ld0.7_Lp0.3_seed1097,[100],0.7,0.3,0.90,1097,0.893059,0.659412,-1.193115,0.594629,...,-0.871363,0.430156,-0.190601,0.476068,-33.667284,0.005241,0.648680,0.607092,-13.802838,0.091986
2009,model_arch100_r0.9_Ld0.7_Lp0.3_seed4916,[100],0.7,0.3,0.90,4916,0.695609,0.724147,-0.669940,0.582423,...,-0.741598,0.543014,-1.005840,0.337188,-32.689610,-0.048360,0.899413,0.671644,-14.312771,0.163738
2010,model_arch100_r0.9_Ld0.7_Lp0.3_seed8685,[100],0.7,0.3,0.90,8685,0.919034,0.663177,-1.416506,0.576585,...,-0.767992,0.437750,0.079239,0.538161,-33.524498,0.026764,0.588635,0.615091,-18.770546,0.012744
2011,model_arch100_r0.9_Ld0.7_Lp0.3_seed6961,[100],0.7,0.3,0.90,6961,0.377737,0.665945,0.200388,0.582231,...,-1.404639,0.427551,-2.303208,0.340812,-32.322118,-0.136000,0.763459,0.627752,-19.193560,-0.038929


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)

SETS_CATEGORY = {
    "SuperZZ1":  "Train",
    "SuperZZ2":  "Val",
    "ZZx1":      "Test",
    "ZZx2":     "Test",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZxReto":  "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
1874,model_arch94_r0.01_Ld0.3_Lp0.7_seed4916,[94],0.878495,-0.659195,-1.764049,-0.747147
1503,model_arch76_r0.01_Ld0.3_Lp0.7_seed7022,[76],0.800662,0.006337,-2.479845,-0.889987
1463,model_arch74_r0.01_Ld0.3_Lp0.7_seed7022,[74],0.816841,-0.266203,-2.347112,-1.018462
1036,model_arch52_r0.9_Ld0.7_Lp0.3_seed3064,[52],0.897334,-0.115785,-2.784721,-1.054136
890,model_arch45_r0.01_Ld0.7_Lp0.3_seed8013,[45],0.830407,0.360477,-3.148202,-1.087764



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_SuperZZ1_theta,R2_SuperZZ2_theta,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZxReto_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
1874,model_arch94_r0.01_Ld0.3_Lp0.7_seed4916,[94],0.878495,-0.659195,-2.742417,0.593767,-4.418370,0.765428,-2.471454,-0.219951,0.720884,-1.823784,-6.280547,0.878495,-0.659195,-1.764049,-0.747147
1503,model_arch76_r0.01_Ld0.3_Lp0.7_seed7022,[76],0.800662,0.006337,-2.669692,0.050861,-9.727704,0.947070,-3.162470,-0.540124,0.747218,-1.699472,-6.264296,0.800662,0.006337,-2.479845,-0.889987
1463,model_arch74_r0.01_Ld0.3_Lp0.7_seed7022,[74],0.816841,-0.266203,-1.518467,0.300797,-14.094973,0.914674,-2.221428,-0.604213,0.046296,-0.575755,-3.370940,0.816841,-0.266203,-2.347112,-1.018462
1036,model_arch52_r0.9_Ld0.7_Lp0.3_seed3064,[52],0.897334,-0.115785,-3.157713,0.747560,-12.552364,0.834469,-3.777216,-1.497168,0.769488,-2.607115,-3.822434,0.897334,-0.115785,-2.784721,-1.054136
890,model_arch45_r0.01_Ld0.7_Lp0.3_seed8013,[45],0.830407,0.360477,-3.218682,0.526083,-14.407016,0.409056,-4.516257,-1.394795,0.631031,-3.047710,-3.315525,0.830407,0.360477,-3.148202,-1.087764


In [5]:
final_table.to_excel("BestModels-1l.xlsx")